In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd

In [77]:
def get_fates_param_dict(fates_pars, fates_key, param_dir, fates_default):

    fates_par_dict = {}
    for parameter in fates_pars:
        
        fates_par_dict[parameter] = {}
        
        param_df = fates_key[fates_key.parameter_name == parameter].copy()
        min_ens = param_df[param_df.type == 'min']['ensemble'].values[0]
        max_ens = param_df[param_df.type == 'max']['ensemble'].values[0]
        
        min_ens_file = os.path.join(param_dir, 'fates_oaat', f"FATES_OAAT_{str(min_ens).rjust(3, '0')}.nc")
        max_ens_file = os.path.join(param_dir, 'fates_oaat', f"FATES_OAAT_{str(max_ens).rjust(3, '0')}.nc")
        
        min_ens_dat = xr.open_dataset(min_ens_file, decode_cf=True, decode_times=False)
        max_ens_dat = xr.open_dataset(max_ens_file, decode_cf=True, decode_times=False)
    
        if parameter in ['fates_stoich_nitr_1', 'fates_stoich_nitr_2',
                        'fates_stoich_nitr_3', 'fates_stoich_nitr_4']:
            param_name = 'fates_stoich_nitr'
        else:
            param_name = parameter
            
        min_da = min_ens_dat[param_name]
        max_da = max_ens_dat[param_name]
        default_da = fates_default[param_name]
    
        if param_name == 'fates_stoich_nitr':
            index = int(parameter.replace(f"{param_name}_", '')) - 1
            min_vals = min_da.values[index]
            max_vals = max_da.values[index]
            default_vals = default_da.values[index]
        else:
            min_vals = min_da.values
            max_vals = max_da.values
            default_vals = default_da.values
    
        if 'fates_pft' in min_da.dims:
            if 'fates_leafage_class' in min_da.dims:
                min_vals = min_vals.flatten()
                max_vals = max_vals.flatten()
                default_vals = default_vals.flatten()
            fates_par_dict[parameter]['type'] = 'pft'
            fates_par_dict[parameter]['min_value'] = min_vals
            fates_par_dict[parameter]['max_value'] = max_vals
            fates_par_dict[parameter]['default_value'] = default_vals
            if (min_vals == min_vals[0]).all() and (max_vals == max_vals[0]).all():
                fates_par_dict[parameter]['pft-specific'] = False
            else: 
                fates_par_dict[parameter]['pft-specific'] = True
    
        elif 'fates_litterclass' in min_da.dims:
            fates_par_dict[parameter]['type'] = 'litterclass'
            fates_par_dict[parameter]['min_value'] = min_vals
            fates_par_dict[parameter]['max_value'] = max_vals
            fates_par_dict[parameter]['default_value'] = default_vals
        elif 'fates_NCWD' in min_da.dims:
            fates_par_dict[parameter]['type'] = 'cwdclass'
            fates_par_dict[parameter]['min_value'] = min_vals
            fates_par_dict[parameter]['max_value'] = max_vals
            fates_par_dict[parameter]['default_value'] = default_vals
        else:
            fates_par_dict[parameter]['type'] = 'global'
            fates_par_dict[parameter]['pft-specific'] = False
            fates_par_dict[parameter]['min_value'] = min_vals
            fates_par_dict[parameter]['max_value'] = max_vals
            fates_par_dict[parameter]['default_value'] = default_vals
            
    return fates_par_dict
    
def get_fates_oaat_df(fates_par_dict, fates_pfts, litter_names, cwd_names):
    records = []
    for param, meta in fates_par_dict.items():
        if param == 'fates_stoich_phos':
            for j in range(4):
                for i, pft in enumerate(fates_pfts):
                    records.append({
                    'parameter': f"{param}_{j}",
                    'index_type': 'pft',
                    'index_name': pft,
                    'min_value': meta['min_value'][j][i],
                    'max_value': meta['max_value'][j][i],
                    'default_value': meta['default_value'][j][i],
                })
        elif meta['type'] == 'global':
            records.append({
                'parameter': param,
                'index_type': 'global',
                'index_name': 'all',
                'min_value': meta['min_value'],
                'max_value': meta['max_value'],
                'default_value': meta['default_value'],
            })
        elif meta['type'] == 'pft':
            for i, pft in enumerate(fates_pfts):
                records.append({
                    'parameter': param,
                    'index_type': 'pft',
                    'index_name': pft,
                    'min_value': (meta['min_value'][i] if not np.isscalar(meta['min_value']) else meta['min_value']),
                    'max_value': (meta['max_value'][i] if not np.isscalar(meta['max_value']) else meta['max_value']),
                    'default_value': meta['default_value'][i],
                })
        elif meta['type'] == 'litterclass':
            for i, lit in enumerate(litter_names):
                records.append({
                    'parameter': param,
                    'index_type': 'litterclass',
                    'index_name': lit,
                    'min_value': meta['min_value'][i],
                    'max_value': meta['max_value'][i],
                    'default_value': meta['default_value'][i],
                })
        elif meta['type'] == 'cwdclass':
            for i, cwd in enumerate(cwd_names):
                records.append({
                    'parameter': param,
                    'index_type': 'cwdclass',
                    'index_name': cwd,
                    'min_value': meta['min_value'][i],
                    'max_value': meta['max_value'][i],
                    'default_value': meta['default_value'][i],
                })
    param_long_df = pd.DataFrame.from_records(records)
    
    return param_long_df

def get_string_list(input_list):
    return [str(s).replace("b'", "").replace("'", "").strip() for s in input_list]

In [2]:
clm_fates_pfts = pd.read_csv('/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/fates_clm_pfts.csv')

param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

clm_default = xr.open_dataset('/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/paramdata/ctsm60_params.c241017.nc',
                             decode_cf=True, decode_times=False)
fates_default = xr.open_dataset(os.path.join(param_dir, 'fates_params_default_sci.1.81.1_api.38.0.0_crops_vai.nc'),
                               decode_cf=True, decode_times=False)

# oaat keys
fates_key = pd.read_csv(os.path.join(param_dir, 'fates_oaat', 'fates_oaat_key.csv'), index_col=0)
clm_key = pd.read_csv(os.path.join(param_dir, 'clm6sp_oaat_key.csv'), header=None)
clm_key.columns = ['ensemble', 'parameter_name', 'type']

In [69]:
# get rid of some FATES parameters
dont_include = ['fates_vcmaxha_clmdef', 'fates_vcmaxhd_clmdef', 'fates_vcmaxse_clmdef',
               'fates_jmaxha_clmdef', 'fates_jmaxhd_clmdef', 'fates_jmaxse_clmdef',
               'fates_new_def', 'default']

fates_key_sub = fates_key[~fates_key.parameter_name.isin(dont_include)].copy()
fates_pars = fates_key_sub.parameter_name.unique()

clm_key_sub = clm_key[~clm_key.parameter_name.isin(dont_include)].copy()
clm_pars = clm_key_sub.parameter_name.unique()

# names of pfts, litter names, etc.
clm_pfts = get_string_list(clm_default.pftname.values)[0:17]
segment_names = get_string_list(clm_default.segment.values)
variant_names = get_string_list(clm_default.variants.values)

fates_pfts = get_string_list(fates_default.fates_pftname.values)
litter_names = get_string_list(fates_default.fates_litterclass_name.values)
cwd_names = ['twigs', 'small_branches', 'large_branches', 'trunks']

In [12]:
fates_par_dict = get_fates_param_dict(fates_pars, fates_key_sub, param_dir,
                                      fates_default)

In [78]:
fates_param_df = get_fates_oaat_df(fates_par_dict, fates_pfts, litter_names,
                                  cwd_names)
fates_param_df.to_csv(os.path.join(param_dir, 'FATES_parameter_SI.csv'))

In [73]:
def get_clm_param_dict(clm_pars, clm_key, param_dir, clm_default):

    clm_par_dict = {}
    for parameter in clm_pars:
        
        clm_par_dict[parameter] = {}
        
        param_df = clm_key[clm_key.parameter_name == parameter].copy()
        min_ens_df = param_df[param_df.type == 'min']
        max_ens_df = param_df[param_df.type == 'max']

        if parameter == 'zetamaxstable':
            clm_par_dict[parameter]['type'] = 'global'
            clm_par_dict[parameter]['pft-specific'] = False
            clm_par_dict[parameter]['min_value'] = 0.1
            clm_par_dict[parameter]['max_value'] = 10.0
            clm_par_dict[parameter]['default_value'] = 2.0
        elif parameter == 'baseflow_scalar':
            clm_par_dict[parameter]['type'] = 'global'
            clm_par_dict[parameter]['pft-specific'] = False
            clm_par_dict[parameter]['min_value'] = 0.0005
            clm_par_dict[parameter]['max_value'] = 0.1
            clm_par_dict[parameter]['default_value'] = 0.001
        elif parameter == 'leaf_mr_vcm':
            clm_par_dict[parameter]['type'] = 'global'
            clm_par_dict[parameter]['pft-specific'] = False
            clm_par_dict[parameter]['min_value'] = 0.012
            clm_par_dict[parameter]['max_value'] = 0.018
            clm_par_dict[parameter]['default_value'] = 0.015
        else:
            if len(min_ens_df) > 0:
                min_ens = min_ens_df['ensemble'].values[0]
                min_ens_file = os.path.join(param_dir, 'clm_oaat', f"{str(min_ens)}.nc")
            else:
                min_ens_file = '/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/paramdata/ctsm60_params.c241017.nc'
    
            if len(max_ens_df) > 0:
                max_ens = max_ens_df['ensemble'].values[0]
                max_ens_file = os.path.join(param_dir, 'clm_oaat', f"{str(max_ens)}.nc")
            else:
                max_ens_file = '/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/paramdata/ctsm60_params.c241017.nc'
    
            min_ens_dat = xr.open_dataset(min_ens_file, decode_cf=True, decode_times=False)
            max_ens_dat = xr.open_dataset(max_ens_file, decode_cf=True, decode_times=False)
        
            min_da = min_ens_dat[parameter]
            max_da = max_ens_dat[parameter]
            default_da = clm_default[parameter]
            
            min_vals = min_da.values
            max_vals = max_da.values
            default_vals = default_da.values
    
            if 'pft' in min_da.dims:
                if 'segment' in min_da.dims:
                    clm_par_dict[parameter]['type'] = 'pft_segment'
                    clm_par_dict[parameter]['min_value'] = min_vals
                    clm_par_dict[parameter]['max_value'] = max_vals
                    clm_par_dict[parameter]['default_value'] = default_vals
                    clm_par_dict[parameter]['pft-specific'] = False
                elif 'variants' in min_da.dims:
                    clm_par_dict[parameter]['type'] = 'pft_variant'
                    clm_par_dict[parameter]['min_value'] = min_vals
                    clm_par_dict[parameter]['max_value'] = max_vals
                    clm_par_dict[parameter]['default_value'] = default_vals
                    clm_par_dict[parameter]['pft-specific'] = False
                else:
                    clm_par_dict[parameter]['type'] = 'pft'
                    clm_par_dict[parameter]['min_value'] = min_vals[0:17]
                    clm_par_dict[parameter]['max_value'] = max_vals[0:17]
                    clm_par_dict[parameter]['default_value'] = default_vals[0:17]
                    if (min_vals == min_vals[0]).all() and (max_vals == max_vals[0]).all():
                        clm_par_dict[parameter]['pft-specific'] = False
                    else: 
                        clm_par_dict[parameter]['pft-specific'] = True
            else:
                clm_par_dict[parameter]['type'] = 'global'
                clm_par_dict[parameter]['pft-specific'] = False
                clm_par_dict[parameter]['min_value'] = min_vals
                clm_par_dict[parameter]['max_value'] = max_vals
                clm_par_dict[parameter]['default_value'] = default_vals
            
    return clm_par_dict

def get_clm_oaat_df(clm_par_dict, clm_pfts, segment_names, variant_names):
    records = []
    for param, meta in clm_par_dict.items():
        if meta['type'] == 'global':
            records.append({
                'parameter': param,
                'index_type': 'global',
                'index_name': 'all',
                'min_value': meta['min_value'],
                'max_value': meta['max_value'],
                'default_value': meta['default_value'],
            })
        elif meta['type'] == 'pft':
            for i, pft in enumerate(clm_pfts):
                records.append({
                    'parameter': param,
                    'index_type': 'pft',
                    'index_name': pft,
                    'min_value': (meta['min_value'][i] if not np.isscalar(meta['min_value']) else meta['min_value']),
                    'max_value': (meta['max_value'][i] if not np.isscalar(meta['max_value']) else meta['max_value']),
                    'default_value': meta['default_value'][i],
                })
        elif meta['type'] == 'pft_segment':
            for j, seg in enumerate(segment_names):
                for i, pft in enumerate(clm_pfts):
                    records.append({
                        'parameter': param,
                        'index_type': 'segment_pft',
                        'index_name': f"{seg}_{pft}",
                        'min_value': (meta['min_value'][j, i] if not np.isscalar(meta['min_value']) else meta['min_value']),
                        'max_value': (meta['max_value'][j, i] if not np.isscalar(meta['max_value']) else meta['max_value']),
                        'default_value': meta['default_value'][j, i],
                    })
        elif meta['type'] == 'pft_variant':
            for j, seg in enumerate(variant_names):
                for i, pft in enumerate(clm_pfts):
                    records.append({
                        'parameter': param,
                        'index_type': 'variant_pft',
                        'index_name': f"{seg}_{pft}",
                        'min_value': (meta['min_value'][j, i] if not np.isscalar(meta['min_value']) else meta['min_value']),
                        'max_value': (meta['max_value'][j, i] if not np.isscalar(meta['max_value']) else meta['max_value']),
                        'default_value': meta['default_value'][j, i],
                    })
    param_long_df = pd.DataFrame.from_records(records)
    
    return param_long_df

In [67]:
clm_par_dict = get_clm_param_dict(clm_pars, clm_key_sub, param_dir, clm_default)

In [74]:
clm_param_df = get_clm_oaat_df(clm_par_dict, clm_pfts, segment_names, variant_names)

In [76]:
clm_param_df.to_csv(os.path.join(param_dir, 'CLM_parameter_SI.csv'))